In [75]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score

In [118]:
def read_file():
    train_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB5/train_dataset.csv'
    train_df = pd.read_csv(train_path)
    return train_df

train_df = read_file()
train_df.shape

(6000, 59)

- shape 6000 x 59
- There are Nan values
- Different types of columns

- build a df copy AND **SPLIT ALREADY INTO TRAIN AND VALIDATION SET**

In [16]:
# do this in the main
def train_eval_split(train_df):
    df = train_df.copy()
    labels = df['target']
    data = df.loc[:, 'cont_0': 'cat_7']

    x_train, x_val, y_train, y_val = train_test_split(data, labels, train_size=0.8, shuffle=True, random_state=42)

    return df, labels, data, x_train, x_val, y_train, y_val

df, labels, data, x_train, x_val, y_train, y_val = train_eval_split(train_df)

- Inspect explicit Nan distribution
- But before doing anything check for weird empty / nan-like values --> explicitly make them nan
- To perform the check divide cols based on their data type

In [ ]:
x_train.info()

In [ ]:
x_train.describe()

- numerical columns need to be standardized

- inspect Nan distribution, ONLY SEE, DON'T MANAGE THEM YET

In [ ]:
print(x_train.isna().mean(axis=0).sort_values(ascending=False)*100)

- all columns have nan % ≈18-21 --> not worth dropping, pretty normal --> impute

In [ ]:
# print(x_train.isna().mean(axis=1).sort_values(ascending=False)*100)

print(x_train.isna().mean(axis=1).sort_values(ascending=False).head(20)*100)
print(x_train.loc[2336, :].isna().sum())

- rows nan % in the whole dataset ≈3-41
    - the worst row has 24 Nan columns out of 59 --> 41%
    - the worst 20 rows have 36% nan

- do I drop the worst 20 rows?
    - what to do is debatable, but I think I'll just keep everything and impute in the preprocessing

- check nan-like values

In [21]:
# do this in the main
num_col = x_train.loc[:, 'cont_0':'cont_29']
ord_col = x_train.loc[:, 'ord_0':'ord_19']
cat_col = x_train.loc[:, 'cat_0':'cat_7']

In [ ]:
for col in num_col:
    print(x_train[col].value_counts().head(5))
    print(x_train[col].value_counts().tail(5))
    print()

- numerical columns need to be standardized

In [ ]:
for col in ord_col:
    print(x_train[col].value_counts().head(5))
    print(x_train[col].value_counts().tail(5))
    print()

- some ordinal columns has only 2 values and quite unbalanced, but they're fine I guess
- no dirty strings / data

In [ ]:
for col in cat_col:
    print(x_train[col].value_counts().head(5))
    print(x_train[col].value_counts().tail(5))
    print()

- no dirty strings / nan like values
- columns cat4,cat_5, cat_6 only has one value --> drop them for the training
    - check also through histplot to be sure

In [ ]:
plt.figure(figsize=(25,25))
for plt_idx,n_c in enumerate(num_col):
    plt.subplot(10,3, plt_idx+1)        # len(x_train[n_c].values)
    sns.histplot(x=x_train[n_c].values, bins='auto', kde=True)
    plt.title(f"Distribution {n_c}")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(20,20))
for plt_idx,o_c in enumerate(ord_col):
    plt.subplot(10,2, plt_idx+1)
    sns.histplot(x=x_train[o_c].values, bins=len(x_train[n_c].values), kde=False)
    plt.title(f"Distribution {o_c}")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,12))
for plt_idx,c_c in enumerate(cat_col):
    plt.subplot(4,2, plt_idx+1)
    sns.histplot(x=x_train[c_c].values, bins=len(x_train[n_c].values), kde=False)
    plt.title(f"Distribution {c_c}")

plt.tight_layout()
plt.show()

- distributions for numerical and ordinal are fine
- drop cat_4,cat_5, cat_6 and also the categorical are fine

In [58]:
# drop useless columns
x_train.drop(columns=['cat_4', 'cat_5', 'cat_6'], inplace=True)

In [63]:
# the categorical columns changed, recompute this
num_col = x_train.loc[:, 'cont_0':'cont_29']
ord_col = x_train.loc[:, 'ord_0':'ord_19']
cat_col = x_train.loc[:, 'cat_0':'cat_7']

num_col_names = num_col.columns.tolist()
ord_col_names = ord_col.columns.tolist()
cat_col_names = cat_col.columns.tolist()


---

# NEXT TIME SPLIT TRAIN AND VALIADTION + JUST LOOK AT NAN DISTRIBUTION (drop if only catastrophic otherwise **impute**) + **QUICKLY** CHECK FOR HIDDEN NANS + HISTPLOT TO DROP USELESS COLUMNS --> THAT'S IT

---


- no weird / hidden nan / nan-like values --> we can manage Nan **BASED ON PREPROCESSING**
- So before let's have an idea of the preprocessing and then decide Nan management:
    - numerical columns: KNNimputer ADD MISSING VALUE FLAG COL + standard scaler
    - ordinal columns: simplimputer, replace nan with 'unknown' and add missing value flag col + ordinal encoder (handle 'unknown' in the OE with -1) --> categories problem.
    - categorical column: simplimputer, replace nan with 'unknown' and add missing value flag col + one hot encoding
- tie all pipelines with ColumnTransformer
- regression model: regression forest --> not KNN regressor because there are a few columns, curse dimensionality

In [71]:
# sanity check
x_train.shape, y_train.shape

((4800, 55), (4800,))

In [112]:
num_pipe = Pipeline(steps=[
    (
        'num_imputer',
        KNNImputer(add_indicator=True, weights='uniform', n_neighbors=5)                # tune: n_neighbors , weights
    ),
    (
        'scaler',
        StandardScaler()
    )
])

ord_pipe = Pipeline(steps=[
    (
        'ord_imputer',
        SimpleImputer(strategy='constant', fill_value='unknown', add_indicator=True)
    )
    ,
    (
        'OE',
        OrdinalEncoder(categories='auto', handle_unknown='use_encoded_value', unknown_value=-1)
    )
])

cat_pipe = Pipeline(steps=[
    (
        'cat_imputer',
        SimpleImputer(strategy='constant', fill_value='unknown', add_indicator=True)
    )
    ,
    (
        'OHE',
        OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    )
])

preprocesing_pipeline = ColumnTransformer(transformers=[
    ('num_pipeline', num_pipe, num_col_names),
    ('ord_pipeline', ord_pipe, ord_col_names),
    ('cat_pipeline', cat_pipe, cat_col_names)],
    remainder='passthrough', n_jobs=-1)

full_pipeline = Pipeline(steps=[
    ('preprocessing', preprocesing_pipeline),
    ('regression', RandomForestRegressor(n_estimators=150, criterion='squared_error', bootstrap=True, n_jobs=-1, random_state=42))        # tune this n_estimators : [100,150], criterion : ['squared_error', 'absolute_error'], max_depth:[5,10,15], 'min_samples_split': [2,5], 'min_samples_leaf':[1,5]
    ])


param_grid = {
    # 'preprocessing__num_pipeline__num_imputer__n_neighbors': [3,5]
    # 'preprocessing__num_pipeline__num_imputer__weights': ['uniform', 'distance'],
    # 'regression__alpha' : [0.5,1,2]
    
    # 'regression__n_estimators' : [100,150],
    # 'regression__criterion' : ['squared_error', 'absolute_error'],
    'regression__min_samples_split' : [5, 10],
    'regression__min_samples_leaf' : [2, 5],
    'regression__max_depth' : [None, 20, 30, 40]      # None, 20, 30, 40
}

grid = GridSearchCV(estimator=full_pipeline, param_grid=param_grid, scoring='r2', n_jobs=-1, cv=4, verbose=1)
grid.fit(X=x_train, y=y_train)
print(grid.best_params_)

best_model = grid.best_estimator_

# get training score and compare with validation score
y_pred_train = best_model.predict(x_train)
print(r2_score(y_pred_train, y_train))

# validation score
y_pred = best_model.predict(x_val)
print(r2_score(y_val, y_pred))

Fitting 4 folds for each of 4 candidates, totalling 16 fits


{'regression__max_depth': None, 'regression__min_samples_leaf': 2, 'regression__min_samples_split': 5}
0.9420894046739847
0.7929423034900873


- a bit of overfitting but i'ts fine :)

- next time just run a standard / default RandomForestRegressor and see the train score + validation score --> if train score >> validation score --> set hyperparameters like n_estimators, min_samples_split, min_samples_leaf, max_depth to post-prune the tree and avoid overfitting.

- Train a default model (or a “reasonable default” RF).
- Check train vs val.

- If train ≈ val and both decent → stop.
- If train >> val → overfit → tune only:
	- max_depth
	- min_samples_leaf
	- min_samples_split
**AND YOU EXPECT THE TRAIN SCORE TO DROP A BIT AND THE VALIDATION TO STAY THE SAME OR IMPROVE**

In [115]:
# move y_pred to its own CSV file
    # convert to DF
    # then to CSV

def write_csv(y_pred):
    y_pred_df = pd.DataFrame({'ID': range(len(y_pred)), 'target': y_pred})
    y_pred_df.to_csv('predictions.csv', index=False)
    
    print(y_pred_df.shape, x_val.shape)

(1200, 2) (1200, 58)


# main.py

- remove train, validation split --> x_train and y_train are the data and labels of train_dataset.csv and x_test and y_test are the data and labels of test_dataset.csv
- don't gridsearch, simply use the found hyperparameters
- remember in the training to return the fitted pipeline --> you need it for the test data
- drop cat_4/5/6 ALSO in test


- OrdinalEncoder “auto” is not ordinal, it’s alphabetical, but it's fine as I don't know how to fix it ...

In [124]:
def read_file_get_data():
    train_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB5/train_dataset.csv'
    test_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB5/test_dataset.csv'
    
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    df = train_df.copy()        # useless but whatever
    y_train = df['target']
    x_train = df.loc[:, 'cont_0': 'cat_7']

    x_test = test_df.loc[:, 'cont_0': 'cat_7']
    
    return df, x_train, y_train, x_test


def split_col_type(x_train):
    x_train.drop(columns=['cat_4', 'cat_5', 'cat_6'], inplace=True)
    num_col = x_train.loc[:, 'cont_0':'cont_29']
    ord_col = x_train.loc[:, 'ord_0':'ord_19']
    cat_col = x_train.loc[:, 'cat_0':'cat_7']

    num_col_names = num_col.columns.tolist()
    ord_col_names = ord_col.columns.tolist()
    cat_col_names = cat_col.columns.tolist()

    return num_col_names, ord_col_names, cat_col_names



def preprocessing_and_training(num_col_names, ord_col_names, cat_col_names, x_train, y_train):
    num_pipe = Pipeline(steps=[
        (
            'num_imputer',
            KNNImputer(add_indicator=True, weights='uniform', n_neighbors=5)                # tune: n_neighbors , weights
        ),
        (
            'scaler',
            StandardScaler()
        )
    ])

    ord_pipe = Pipeline(steps=[
        (
            'ord_imputer',
            SimpleImputer(strategy='constant', fill_value='unknown', add_indicator=True)
        )
        ,
        (
            'OE',
            OrdinalEncoder(categories='auto', handle_unknown='use_encoded_value', unknown_value=-1)
        )
    ])

    cat_pipe = Pipeline(steps=[
        (
            'cat_imputer',
            SimpleImputer(strategy='constant', fill_value='unknown', add_indicator=True)
        )
        ,
        (
            'OHE',
            OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        )
    ])

    preprocesing_pipeline = ColumnTransformer(transformers=[
        ('num_pipeline', num_pipe, num_col_names),
        ('ord_pipeline', ord_pipe, ord_col_names),
        ('cat_pipeline', cat_pipe, cat_col_names)],
        remainder='passthrough', n_jobs=-1)


    full_pipeline = Pipeline(steps=[
        ('preprocessing', preprocesing_pipeline),
        ('regression', RandomForestRegressor(n_estimators=150, max_depth=None, min_samples_leaf=2, min_samples_split=5, criterion='squared_error', bootstrap=True, n_jobs=-1, random_state=42))        # tune this n_estimators : [100,150], criterion : ['squared_error', 'absolute_error'], max_depth:[5,10,15], 'min_samples_split': [2,5], 'min_samples_leaf':[1,5]
        ])

    full_pipeline.fit(x_train, y_train)
    
    y_pred_train = full_pipeline.predict(x_train)
    print(f'training score: {r2_score(y_train, y_pred_train)}')

    return full_pipeline



def get_predictions(full_pipeline, x_test):
    x_test.drop(columns=['cat_4', 'cat_5', 'cat_6'], inplace=True)

    y_pred = full_pipeline.predict(x_test)
    y_pred_df = pd.DataFrame({'ID': range(len(y_pred)), 'target': y_pred})  # we were not given an ID column, so it I'll use a range, BUT IF THERE WAS ONE YOU WERE SUPPOSED TO USE IT
    y_pred_df.to_csv('predictions.csv', index=False)



if __name__ == '__main__':
    df, x_train, y_train, x_test = read_file_get_data()

    num_col_names, ord_col_names, cat_col_names = split_col_type(x_train)

    full_pipeline = preprocessing_and_training(num_col_names, ord_col_names, cat_col_names, x_train, y_train)

    get_predictions(full_pipeline, x_test)


training score: 0.9575394812551954


- add validation data to the training data
- (transform +) predict test data
- build CSV
- move everything to main.py

(- possible additional step to the pipeline: polynomial features --> you need to figure on which columns with linear and non-linear correlation)